In [7]:
%pip -q install pandas numpy scikit-learn matplotlib seaborn joblib boto3 sagemaker ucimlrepo

Note: you may need to restart the kernel to use updated packages.


In [8]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
wine_quality = fetch_ucirepo(id=186) 
  
# data (as pandas dataframes) 
X = wine_quality.data.features 
y = wine_quality.data.targets 
  
# metadata 
print(wine_quality.metadata) 
  
# variable information 
print(wine_quality.variables) 


{'uci_id': 186, 'name': 'Wine Quality', 'repository_url': 'https://archive.ics.uci.edu/dataset/186/wine+quality', 'data_url': 'https://archive.ics.uci.edu/static/public/186/data.csv', 'abstract': 'Two datasets are included, related to red and white vinho verde wine samples, from the north of Portugal. The goal is to model wine quality based on physicochemical tests (see [Cortez et al., 2009], http://www3.dsi.uminho.pt/pcortez/wine/).', 'area': 'Business', 'tasks': ['Classification', 'Regression'], 'characteristics': ['Multivariate'], 'num_instances': 4898, 'num_features': 11, 'feature_types': ['Real'], 'demographics': [], 'target_col': ['quality'], 'index_col': None, 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 2009, 'last_updated': 'Wed Nov 15 2023', 'dataset_doi': '10.24432/C56S3T', 'creators': ['Paulo Cortez', 'A. Cerdeira', 'F. Almeida', 'T. Matos', 'J. Reis'], 'intro_paper': {'ID': 252, 'type': 'NATIVE', 'title': 'Modeling wine preferences

In [9]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import boto3
import sagemaker

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sagemaker import get_execution_role
from sagemaker.sklearn.estimator import SKLearn
from sagemaker.sklearn.model import SKLearnModel

pd.set_option("display.max_columns", 200)
sns.set_theme(style="whitegrid")

print("Setup complete ✅")
print("pandas:", pd.__version__)
print("numpy:", np.__version__)

Setup complete ✅
pandas: 2.3.3
numpy: 1.26.4


In [10]:
# Combining both datasets

df = wine_quality.data.features.copy()
df["quality"] = wine_quality.data.targets.iloc[:, 0].astype(float)
df["wine_type"] = wine_quality.data.original["color"].astype(str).values

print(df.shape)
print(df["wine_type"].value_counts())
df.head()

(6497, 13)
wine_type
white    4898
red      1599
Name: count, dtype: int64


,fixed_acidity,volatile_acidity,citric_acid,residual_sugar,chlorides,free_sulfur_dioxide,total_sulfur_dioxide,density,pH,sulphates,alcohol,quality,wine_type
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5.0,red
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5.0,red
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5.0,red
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6.0,red
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5.0,red


In [11]:
#Limited the input/output values for deployment

loaded = joblib.load("artifacts/wine_lr.joblib")
if isinstance(loaded, dict) and "features" in loaded:
    FEATURES = loaded["features"]
else:
    FEATURES = getattr(loaded, "feature_names_in_", None)
    if FEATURES is None:
        raise ValueError("Couldn't infer FEATURES from model artifact.")

FEATURES = list(FEATURES)

df_bounds = df.copy()
df_bounds.columns = (
    df_bounds.columns.astype(str)
    .str.strip()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

missing = [c for c in FEATURES if c not in df_bounds.columns]
if missing:
    raise ValueError(f"These model feature columns are missing from df: {missing}")

X_bounds = df_bounds[FEATURES].astype(float)

bounds = {}
for c in FEATURES:
    s = X_bounds[c]
    bounds[c] = {
        "min": float(s.min()),
        "max": float(s.max()),
        "p01": float(s.quantile(0.01)),
        "p99": float(s.quantile(0.99)),
    }

os.makedirs("artifacts", exist_ok=True)
with open("artifacts/feature_bounds.json", "w") as f:
    json.dump(bounds, f, indent=2)

print("Wrote artifacts/feature_bounds.json for", len(bounds), "features.")
print("Example bounds:", list(bounds.items())[:2])

Wrote artifacts/feature_bounds.json for 11 features.
Example bounds: [('fixed_acidity', {'min': 3.8, 'max': 15.9, 'p01': 5.1, 'p99': 12.0}), ('volatile_acidity', {'min': 0.08, 'max': 1.58, 'p01': 0.12, 'p99': 0.88})]


In [12]:
#Placing bounds on json

assert "df" in globals(), "df is not defined. Run the cell that creates df first."

model_artifact = "/home/sagemaker-user/shared/heroku_app/artifacts/wine_lr.joblib"
loaded = joblib.load(model_artifact)
FEATURES = list(loaded["features"]) if isinstance(loaded, dict) and "features" in loaded else None
if not FEATURES:
    raise ValueError("Could not read FEATURES from the model artifact.")

df_bounds = df.copy()
df_bounds.columns = (
    df_bounds.columns.astype(str)
    .str.strip()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

missing = [c for c in FEATURES if c not in df_bounds.columns]
if missing:
    raise ValueError(f"These model feature columns are missing from df: {missing}")

X_bounds = df_bounds[FEATURES].astype(float)

bounds = {}
for c in FEATURES:
    s = X_bounds[c]
    bounds[c] = {
        "min": float(s.min()),
        "max": float(s.max()),
        "p01": float(s.quantile(0.01)),
        "p99": float(s.quantile(0.99)),
    }

out_path = "/home/sagemaker-user/shared/heroku_app/artifacts/feature_bounds.json"
os.makedirs(os.path.dirname(out_path), exist_ok=True)
with open(out_path, "w") as f:
    json.dump(bounds, f, indent=2)

print("WROTE:", out_path)
print("Example:", list(bounds.items())[:2])

WROTE: /home/sagemaker-user/shared/heroku_app/artifacts/feature_bounds.json
Example: [('fixed_acidity', {'min': 3.8, 'max': 15.9, 'p01': 5.1, 'p99': 12.0}), ('volatile_acidity', {'min': 0.08, 'max': 1.58, 'p01': 0.12, 'p99': 0.88})]


In [13]:
# Defining Features and Test/Train

FEATURES = [
    "fixed_acidity",
    "volatile_acidity",
    "citric_acid",
    "residual_sugar",
    "chlorides",
    "free_sulfur_dioxide",
    "total_sulfur_dioxide",
    "density",
    "pH",
    "sulphates",
    "alcohol",
]

X = df[FEATURES].copy()
y = df["quality"].astype(float).copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("X_train:", X_train.shape, "X_test:", X_test.shape)
print("y_train:", y_train.shape, "y_test:", y_test.shape)
print("\nTarget summary (train):")
print(y_train.describe())

X_train: (5197, 11) X_test: (1300, 11)
y_train: (5197,) y_test: (1300,)

Target summary (train):
count    5197.000000
mean        5.814508
std         0.876648
min         3.000000
25%         5.000000
50%         6.000000
75%         6.000000
max         9.000000
Name: quality, dtype: float64


In [14]:
# Linear Regression Model

lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred = lr.predict(X_test)

mse  = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae  = mean_absolute_error(y_test, y_pred)
r2   = r2_score(y_test, y_pred)

print(f"RMSE: {rmse:.4f}")
print(f"MAE : {mae:.4f}")
print(f"R^2 : {r2:.4f}")

RMSE: 0.7394
MAE : 0.5659
R^2 : 0.2598


In [15]:
# Actual vs Prediction 

check = pd.DataFrame({
    "actual": y_test.values,
    "pred": y_pred,
})
check["error"] = check["pred"] - check["actual"]

check.head(10)

,actual,pred,error
0,8.0,6.623020,-1.376980
1,5.0,5.015842,0.015842
2,7.0,6.285818,-0.714182
3,6.0,5.699531,-0.300469
4,6.0,5.391815,-0.608185
5,6.0,6.195437,0.195437
6,5.0,6.054269,1.054269
7,6.0,6.089681,0.089681
8,5.0,5.113698,0.113698
9,7.0,6.323571,-0.676429


In [16]:
print("Error Report:")
print(check["error"].describe())
print("\nMean abs error (from table):", check["error"].abs().mean())

Error Report:
count    1300.000000
mean       -0.024448
std         0.739269
min        -2.442260
25%        -0.520178
50%         0.006706
75%         0.398951
max         3.970042
Name: error, dtype: float64

Mean abs error (from table): 0.5658710079723461


In [17]:
# Saving as joblib

os.makedirs("artifacts", exist_ok=True)

joblib.dump(
    {"model": lr, "features": FEATURES},
    "artifacts/wine_lr.joblib"
)

print("Saved: artifacts/wine_lr.joblib")

Saved: artifacts/wine_lr.joblib


In [18]:
# Making sure it's good to go

artifact = joblib.load("artifacts/wine_lr.joblib")
model = artifact["model"]
features = artifact["features"]

one = X_test.iloc[[0]][features]
pred_one = float(model.predict(one)[0])

print("Input (first test row):")
display(one)

print("Prediction:", pred_one)
print("Actual    :", float(y_test.iloc[0]))

Input (first test row):


,fixed_acidity,volatile_acidity,citric_acid,residual_sugar,chlorides,free_sulfur_dioxide,total_sulfur_dioxide,density,pH,sulphates,alcohol
3103,7.0,0.17,0.74,12.8,0.045,24.0,126.0,0.9942,3.26,0.38,12.2


Prediction: 6.623020438300564
Actual    : 8.0


In [19]:
# Uploading data to S3

sess = sagemaker.Session()
role = get_execution_role()
bucket = sess.default_bucket()
prefix = "wine-quality-lr"

train_df = df.loc[X_train.index, FEATURES + ["quality"]]
test_df  = df.loc[X_test.index,  FEATURES + ["quality"]]

os.makedirs("sm_data", exist_ok=True)
train_path = "sm_data/train.csv"
test_path  = "sm_data/test.csv"

train_df.to_csv(train_path, index=False)
test_df.to_csv(test_path, index=False)

s3_train = sess.upload_data(train_path, bucket=bucket, key_prefix=f"{prefix}/train")
s3_test  = sess.upload_data(test_path,  bucket=bucket, key_prefix=f"{prefix}/test")

print("bucket :", bucket)
print("s3_train:", s3_train)
print("s3_test :", s3_test)

sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
bucket : amazon-sagemaker-357954864016-us-east-2-4lb4kae9iij0o9
s3_train: s3://amazon-sagemaker-357954864016-us-east-2-4lb4kae9iij0o9/wine-quality-lr/train/train.csv
s3_test : s3://amazon-sagemaker-357954864016-us-east-2-4lb4kae9iij0o9/wine-quality-lr/test/test.csv


In [53]:
# Creating my training set without a container 

os.makedirs("src", exist_ok=True)

train_py = """\
import os
import joblib
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

def main():
    train_df = pd.read_csv("/opt/ml/input/data/train/train.csv")
    test_df  = pd.read_csv("/opt/ml/input/data/test/test.csv")

    X_train = train_df.drop(columns=["quality"])
    y_train = train_df["quality"].astype(float)

    X_test = test_df.drop(columns=["quality"])
    y_test = test_df["quality"].astype(float)

    model = LinearRegression().fit(X_train, y_train)
    y_pred = model.predict(X_test)

    rmse = float(np.sqrt(mean_squared_error(y_test, y_pred)))
    mae  = float(mean_absolute_error(y_test, y_pred))
    r2   = float(r2_score(y_test, y_pred))

    print(f"RMSE={rmse:.6f}")
    print(f"MAE={mae:.6f}")
    print(f"R2={r2:.6f}")

    os.makedirs("/opt/ml/model", exist_ok=True)
    joblib.dump(model, "/opt/ml/model/model.joblib")

if __name__ == "__main__":
    main()
"""

with open("src/train.py", "w", encoding="utf-8") as f:
    f.write(train_py)

print("Wrote: src/train.py")

Wrote: src/train.py


In [54]:
# Running training dataset

output_path = f"s3://{bucket}/{prefix}/output"

est = SKLearn(
    entry_point="train.py",
    source_dir="src",
    role=role,
    instance_count=1,
    instance_type="ml.m5.large",
    framework_version="1.2-1",
    py_version="py3",
    output_path=output_path,
)

est.fit({"train": s3_train, "test": s3_test})

sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
sagemaker.config INFO - Applied value from config key = SageMaker.TrainingJob.Environment
2026-02-23 01:48:24 Starting - Starting the training job...
2026-02-23 01:48:38 Starting - Preparing the instances for training...
2026-02-23 01:49:24 Downloading - Downloading the training image......../miniconda3/lib/python3.9/site-packages/sagemaker_containers/_server.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
2026-02-23 01:50:32,848 sagemaker-containers INFO     Imported framework sagemaker_sklearn_container.training
2026-02-23 01:50:32,852 sagemak

In [55]:
# Had to create an inference markdown due to an error.

inference_py = """\
import os
import json
import joblib
import numpy as np

def model_fn(model_dir):
    return joblib.load(os.path.join(model_dir, "model.joblib"))

def input_fn(request_body, request_content_type):
    if request_content_type != "text/csv":
        raise ValueError(f"Unsupported content type: {request_content_type}")

    lines = request_body.strip().splitlines()
    rows = [np.fromstring(line, sep=",", dtype=float) for line in lines]

    if len(rows) == 1:
        return rows[0].reshape(1, -1)  # always 2D
    return np.vstack(rows)

def predict_fn(data, model):
    return model.predict(data)

def output_fn(prediction, accept):
    body = json.dumps({"prediction": [float(x) for x in np.ravel(prediction)]})
    return body, "application/json"
"""

with open("src/inference.py", "w", encoding="utf-8") as f:
    f.write(inference_py)

print("Wrote: src/inference.py")

Wrote: src/inference.py


In [56]:
# Deploy Endpoint

model = SKLearnModel(
    model_data=est.model_data,
    role=role,
    entry_point="inference.py",
    source_dir="src",
    framework_version="1.2-1",
    py_version="py3",
)

predictor = model.deploy(initial_instance_count=1, instance_type="ml.m5.large")
print("endpoint:", predictor.endpoint_name)

sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
------!endpoint: sagemaker-scikit-learn-2026-02-23-01-51-52-818


In [57]:
# Testing

predictor.serializer = sagemaker.serializers.CSVSerializer()
predictor.deserializer = sagemaker.deserializers.JSONDeserializer()

sample = X_test.iloc[[0]]
pred = predictor.predict(sample.values)

print("prediction:", pred)
print("actual    :", float(y_test.iloc[0]))

prediction: {'prediction': [6.6230204383005855]}
actual    : 8.0


In [62]:
# Creating my training set with a container 

os.makedirs("byoc_train", exist_ok=True)

dockerfile = """\
FROM python:3.11-slim

RUN pip install --no-cache-dir -q pandas numpy scikit-learn joblib

WORKDIR /opt/program
COPY train.py /opt/program/train.py

ENV PYTHONUNBUFFERED=1
ENTRYPOINT ["python", "/opt/program/train.py"]
"""

train_py = """\
import os
import joblib
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

def main():
    train_df = pd.read_csv("/opt/ml/input/data/train/train.csv")
    test_df  = pd.read_csv("/opt/ml/input/data/test/test.csv")

    X_train = train_df.drop(columns=["quality"])
    y_train = train_df["quality"].astype(float)

    X_test = test_df.drop(columns=["quality"])
    y_test = test_df["quality"].astype(float)

    model = LinearRegression().fit(X_train, y_train)
    y_pred = model.predict(X_test)

    rmse = float(np.sqrt(mean_squared_error(y_test, y_pred)))
    mae  = float(mean_absolute_error(y_test, y_pred))
    r2   = float(r2_score(y_test, y_pred))

    print(f"RMSE={rmse:.6f}")
    print(f"MAE={mae:.6f}")
    print(f"R2={r2:.6f}")

    os.makedirs("/opt/ml/model", exist_ok=True)
    joblib.dump(model, "/opt/ml/model/model.joblib")

if __name__ == "__main__":
    main()
"""

with open("byoc_train/Dockerfile", "w", encoding="utf-8") as f:
    f.write(dockerfile)

with open("byoc_train/train.py", "w", encoding="utf-8") as f:
    f.write(train_py)

print("created:")
print(" - byoc_train/Dockerfile")
print(" - byoc_train/train.py")

created:
 - byoc_train/Dockerfile
 - byoc_train/train.py


In [63]:
# Repo and Image

repo = "wine-quality-lr-byoc-train"
region = boto3.session.Session().region_name
account = boto3.client("sts").get_caller_identity()["Account"]
image_uri = f"{account}.dkr.ecr.{region}.amazonaws.com/{repo}:latest"

print("repo     :", repo)
print("region   :", region)
print("account  :", account)
print("image_uri:", image_uri)

repo     : wine-quality-lr-byoc-train
region   : us-east-2
account  : 357954864016
image_uri: 357954864016.dkr.ecr.us-east-2.amazonaws.com/wine-quality-lr-byoc-train:latest


In [65]:
# Train ECR Image

region = boto3.session.Session().region_name
account = boto3.client("sts").get_caller_identity()["Account"]

image_uri = f"{account}.dkr.ecr.{region}.amazonaws.com/wine-quality-lr-byoc-train:latest"
output_path = f"s3://{bucket}/{prefix}/byoc-output"

byoc_est = sagemaker.estimator.Estimator(
    image_uri=image_uri,
    role=role,
    instance_count=1,
    instance_type="ml.m5.large",
    output_path=output_path,
    sagemaker_session=sess,
)

byoc_est.fit({"train": s3_train, "test": s3_test})

sagemaker.config INFO - Applied value from config key = SageMaker.TrainingJob.Environment
2026-02-23 02:52:28 Starting - Starting the training job...
2026-02-23 02:53:03 Downloading - Downloading input data...
2026-02-23 02:53:28 Training - Training image download completed. Training in progress..RMSE=0.739389
MAE=0.565871
R2=0.259767

2026-02-23 02:53:46 Uploading - Uploading generated training model
2026-02-23 02:53:46 Completed - Training job completed
Training seconds: 44
Billable seconds: 44


In [66]:
# Confirm model

desc = byoc_est.latest_training_job.describe()
print("status:", desc["TrainingJobStatus"])
print("model artifact:", desc["ModelArtifacts"]["S3ModelArtifacts"])
if desc["TrainingJobStatus"] == "Failed":
    print("reason:", desc.get("FailureReason"))

status: Completed
model artifact: s3://amazon-sagemaker-357954864016-us-east-2-4lb4kae9iij0o9/wine-quality-lr/byoc-output/wine-quality-lr-byoc-train-2026-02-23-02-52-26-333/output/model.tar.gz


In [67]:
# Train Container

region = boto3.session.Session().region_name
account = boto3.client("sts").get_caller_identity()["Account"]

image_uri = f"{account}.dkr.ecr.{region}.amazonaws.com/wine-quality-lr-byoc-train:latest"
output_path = f"s3://{bucket}/{prefix}/byoc-output"

byoc_est = sagemaker.estimator.Estimator(
    image_uri=image_uri,
    role=role,
    instance_count=1,
    instance_type="ml.m5.large",
    output_path=output_path,
    sagemaker_session=sess,
)

byoc_est.fit({"train": s3_train, "test": s3_test})

sagemaker.config INFO - Applied value from config key = SageMaker.TrainingJob.Environment
2026-02-23 02:56:46 Starting - Starting the training job...
2026-02-23 02:57:00 Starting - Preparing the instances for training...
2026-02-23 02:57:47 Downloading - Downloading the training image
2026-02-23 02:57:47 Training - Training image download completed. Training in progress..RMSE=0.739389
MAE=0.565871
R2=0.259767

2026-02-23 02:58:11 Uploading - Uploading generated training model
2026-02-23 02:58:11 Completed - Training job completed
Training seconds: 48
Billable seconds: 48


In [68]:
# Confirm Container

desc = byoc_est.latest_training_job.describe()
print("status:", desc["TrainingJobStatus"])
print("model artifact:", desc["ModelArtifacts"]["S3ModelArtifacts"])
if desc["TrainingJobStatus"] == "Failed":
    print("reason:", desc.get("FailureReason"))

status: Completed
model artifact: s3://amazon-sagemaker-357954864016-us-east-2-4lb4kae9iij0o9/wine-quality-lr/byoc-output/wine-quality-lr-byoc-train-2026-02-23-02-56-43-647/output/model.tar.gz


In [69]:
# Checking the model works 

job_name = byoc_est.latest_training_job.name
desc = byoc_est.latest_training_job.describe()

print("job:", job_name)
print("status:", desc["TrainingJobStatus"])
print("model_artifact:", desc["ModelArtifacts"]["S3ModelArtifacts"])

if desc["TrainingJobStatus"] == "Failed":
    print("reason:", desc.get("FailureReason"))

job: wine-quality-lr-byoc-train-2026-02-23-02-56-43-647
status: Completed
model_artifact: s3://amazon-sagemaker-357954864016-us-east-2-4lb4kae9iij0o9/wine-quality-lr/byoc-output/wine-quality-lr-byoc-train-2026-02-23-02-56-43-647/output/model.tar.gz


In [70]:
#Training Log

byoc_est.sagemaker_session.logs_for_job(job_name, wait=False)

2026-02-23 02:58:11 Starting - Preparing the instances for training
2026-02-23 02:58:11 Downloading - Downloading the training image
2026-02-23 02:58:11 Training - Training image download completed. Training in progress.
2026-02-23 02:58:11 Uploading - Uploading generated training model
2026-02-23 02:58:11 Completed - Training job completedRMSE=0.739389
MAE=0.565871
R2=0.259767


In [71]:
# Download and Test

local_dir = "byoc_artifact"
os.makedirs(local_dir, exist_ok=True)

model_s3 = desc["ModelArtifacts"]["S3ModelArtifacts"]
sagemaker.s3.S3Downloader.download(model_s3, local_dir)

import tarfile, glob
tar_path = glob.glob(os.path.join(local_dir, "*.tar.gz"))[0]
with tarfile.open(tar_path, "r:gz") as tar:
    tar.extractall(local_dir)

model = joblib.load(os.path.join(local_dir, "model.joblib"))
pred = float(model.predict(X_test.iloc[[0]].values)[0])

print("prediction:", pred)
print("actual    :", float(y_test.iloc[0]))

sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
prediction: 6.6230204383005855
actual    : 8.0


In [75]:
#Creating Flask API files

import os, shutil

APP_DIR = "heroku_app"
ARTIFACT_SRC = "artifacts/wine_lr.joblib"
ARTIFACT_DST_DIR = os.path.join(APP_DIR, "artifacts")
ARTIFACT_DST = os.path.join(ARTIFACT_DST_DIR, "wine_lr.joblib")

os.makedirs(APP_DIR, exist_ok=True)
os.makedirs(ARTIFACT_DST_DIR, exist_ok=True)

app_py = """\
import os
import joblib
import numpy as np
from flask import Flask, request, jsonify

app = Flask(__name__)

FEATURES = [
    "fixed_acidity",
    "volatile_acidity",
    "citric_acid",
    "residual_sugar",
    "chlorides",
    "free_sulfur_dioxide",
    "total_sulfur_dioxide",
    "density",
    "pH",
    "sulphates",
    "alcohol",
]

MODEL_PATH = os.path.join(os.path.dirname(__file__), "artifacts", "wine_lr.joblib")

_loaded = joblib.load(MODEL_PATH)
if isinstance(_loaded, dict) and "model" in _loaded:
    model = _loaded["model"]
else:
    model = _loaded

@app.get("/health")
def health():
    return jsonify({"status": "ok"})

@app.post("/predict")
def predict():
    data = request.get_json(silent=True) or {}
    missing = [f for f in FEATURES if f not in data]
    if missing:
        return jsonify({"error": "missing_fields", "missing": missing}), 400

    x = np.array([[float(data[f]) for f in FEATURES]], dtype=float)
    pred = float(model.predict(x)[0])
    return jsonify({"prediction": pred})
"""

with open(os.path.join(APP_DIR, "app.py"), "w", encoding="utf-8") as f:
    f.write(app_py)

req = """\
flask==3.0.3
gunicorn==22.0.0
joblib==1.4.2
numpy==1.26.4
scikit-learn==1.7.2
"""
with open(os.path.join(APP_DIR, "requirements.txt"), "w", encoding="utf-8") as f:
    f.write(req)

with open(os.path.join(APP_DIR, "Procfile"), "w", encoding="utf-8") as f:
    f.write("web: gunicorn app:app\n")

dockerfile = """\
FROM python:3.11-slim

WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY app.py /app/app.py
COPY artifacts /app/artifacts

ENV PORT=8080
CMD ["sh", "-c", "gunicorn -b 0.0.0.0:$PORT app:app"]
"""
with open(os.path.join(APP_DIR, "Dockerfile"), "w", encoding="utf-8") as f:
    f.write(dockerfile)

with open(os.path.join(APP_DIR, ".dockerignore"), "w", encoding="utf-8") as f:
    f.write("__pycache__/\n*.pyc\n.ipynb_checkpoints/\n")

if not os.path.exists(ARTIFACT_SRC):
    raise FileNotFoundError(f"Missing {ARTIFACT_SRC}. Make sure you created it earlier.")
shutil.copy2(ARTIFACT_SRC, ARTIFACT_DST)

print("Created Heroku app at:", APP_DIR)
print("Model copied to:", ARTIFACT_DST)
print("Files:", os.listdir(APP_DIR))

Created Heroku app at: heroku_app
Model copied to: heroku_app/artifacts/wine_lr.joblib
Files: ['.git', '.ipynb_checkpoints', '__pycache__', 'artifacts', 'heroku.yml', '.dockerignore', '.gitignore', 'Dockerfile', 'Procfile', 'app.py', 'requirements.txt']
